# Day 9 v2 — Silver: Invoice Metadata (For Loop Pattern)

**Same logic as v1 but wrapped in an entity config list.**

Invoices is the only blob-batch entity here, but the pattern is ready for
future batch entities (e.g. payment_receipts, contracts).

**What changed from v1:**
- Entity config list with natural_key and date_cols
- Wrapped in a for loop
- Full overwrite (incremental MERGE is v3)

**Bronze source:** `bronze-volume/invoices/_metadata/` (Delta)
**Silver sink:** `silver-volume/invoices/` (Delta)

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_BATCH_BASE = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
SILVER_BATCH_BASE = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze base : {BRONZE_BATCH_BASE}")
print(f"Silver base : {SILVER_BATCH_BASE}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Entity config for blob batch entities
# Bronze format: Delta metadata table (invoices store PDFs; Silver = metadata only)
# silver_date_cols: columns that need to be assembled from year/month/day string parts

ENTITIES = [
    {
        "entity_name"   : "invoices",
        "natural_key"   : "invoice_id",
        "bronze_source" : "invoices/_metadata",   # Delta table path under bronze-volume
        "silver_path"   : "invoices",
        "cast": {
            "invoice_id"  : "string",
            "year"        : "integer",
            "month"       : "integer",
            "day"         : "integer",
            "file_size_kb": "decimal(10,2)",
            "bronze_path" : "string",
        },
        # Columns assembled from year/month/day after cast
        "date_col"   : "invoice_date",    # new date column to create
        "year_col"   : "year",
        "month_col"  : "month",
        "day_col"    : "day",
    },
]

print(f"Batch entities configured: {len(ENTITIES)}")
for e in ENTITIES:
    print(f"  {e['entity_name']:<20} key={e['natural_key']}  bronze={e['bronze_source']}")

In [ ]:
# Cell 4 — For loop: transform all batch entities

results = []

for entity in ENTITIES:
    name        = entity["entity_name"]
    natural_key = entity["natural_key"]
    cast_map    = entity["cast"]
    bronze_path = f"{BRONZE_BATCH_BASE}/{entity['bronze_source']}"
    silver_path = f"{SILVER_BATCH_BASE}/{entity['silver_path']}"

    print(f"Processing: {name} ...", end=" ")

    try:
        # Step 1: Read Bronze (Delta for batch entities)
        raw_df = spark.read.format("delta").load(bronze_path)
        bronze_rows = raw_df.count()

        # Step 2: Cast columns
        cast_exprs = [
            F.col(c).cast(t).alias(c)
            for c, t in cast_map.items()
            if c in raw_df.columns
        ]
        typed_df = raw_df.select(cast_exprs)

        # Step 3: Build invoice_date from year/month/day integer columns
        if entity.get("date_col"):
            typed_df = typed_df.withColumn(
                entity["date_col"],
                F.to_date(
                    F.concat_ws("-",
                        F.col(entity["year_col"]).cast("string"),
                        F.lpad(F.col(entity["month_col"]).cast("string"), 2, "0"),
                        F.lpad(F.col(entity["day_col"]).cast("string"), 2, "0"),
                    ), "yyyy-MM-dd"
                )
            ).withColumnRenamed("year",  "invoice_year") \
             .withColumnRenamed("month", "invoice_month") \
             .withColumnRenamed("day",   "invoice_day")

        # Step 4: Trim string columns
        for col_name, col_type in typed_df.dtypes:
            if col_type == "string":
                typed_df = typed_df.withColumn(col_name, F.trim(F.col(col_name)))

        # Step 5: Add audit columns
        audited_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit("full"))
            .withColumn("silver_pipeline",    F.lit("pl_silver_blob_invoices_v2"))
        )

        # Step 6: Deduplicate on natural_key
        window = Window.partitionBy(natural_key).orderBy(F.col("silver_ingested_at").desc())
        deduped_df = (
            audited_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        silver_rows = deduped_df.count()

        # Step 7: Write to Silver Delta (full overwrite)
        (
            deduped_df.write.format("delta")
            .mode("overwrite").option("overwriteSchema", "true")
            .save(silver_path)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"entity": name, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as ex:
        print(f"FAILED")
        results.append({"entity": name, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(ex)})

print("\nAll batch entities done.")

In [ ]:
# Cell 5 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 60)
print("SILVER BLOB INVOICES v2 — RUN SUMMARY")
print("=" * 60)
print(f"  run_timestamp : {RUN_TS}")
print(f"  succeeded     : {len(succeeded)}")
print(f"  failed        : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['entity']:<20} bronze={r['bronze_rows']:>8}  silver={r['silver_rows']:>8}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")